# Notebook 03 — Multi-Task CEIS Fine-Tuning

Fine-tune LAYA with multi-task loss on MedNLI.

### Multi-Task Objective:
- **Verdict (choice, 3-way)**: `supported`, `unsupported`, `contradicted` (Weight: 0.60)
- **Explicitly Mentioned (yes/no)**: Binary presence flag (Weight: 0.20)
- **Needs More Evidence (yes/no)**: Evidence sufficiency flag (Weight: 0.20)

Starting checkpoint: `pavanmantha/mednli_laya`

In [ ]:
import sys, os, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))

from training.trainer import CEISTrainer, TrainSettings
from training.schema import CEISSchema

## 1 · Configure Training Parameters

In [ ]:
# Load settings from YAML
config_path = Path('../config/train_config.yaml')
settings = TrainSettings.from_yaml(config_path, overrides={
    'out': '../runs/ceis_notebook_run',
    'epochs': 2,
    'limit': 150,  # fast demonstration run
})
print('Configured Settings:')
print(f'  Base Checkpoint : {settings.base}')
print(f'  Target Epochs   : {settings.epochs}')
print(f'  Batch Size      : {settings.batch} (Accumulation: {settings.grad_accum})')
print(f'  Device          : {settings.device}')

## 2 · Execute Multi-Task Training Loop

In [ ]:
trainer = CEISTrainer(settings)
print('Starting fine-tuning...')
train_results = trainer.run()
print('\nTraining Complete!')
print('Best Epoch Results:', json.dumps(train_results.get('best', {}), indent=2))

## 3 · Plot Training & Validation Trajectory

In [ ]:
history = train_results.get('history', [])
if history:
    hist_df = pd.DataFrame(history)
    display(hist_df)
    
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].plot(hist_df['epoch'], hist_df['train_loss'], marker='o', color='purple')
    ax[0].set_title('Training Loss')
    ax[0].set_xlabel('Epoch')
    ax[0].set_ylabel('Loss')
    ax[0].grid(True)
    
    ax[1].plot(hist_df['epoch'], hist_df['macro_f1'], marker='s', color='green', label='Macro F1')
    ax[1].plot(hist_df['epoch'], hist_df['accuracy'], marker='^', color='blue', label='Accuracy')
    ax[1].set_title('Validation Performance')
    ax[1].set_xlabel('Epoch')
    ax[1].legend()
    ax[1].grid(True)
    plt.tight_layout()
    plt.show()